In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from collections import defaultdict

In [52]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0"
    ),
    "Accept": ("text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"),
    "Accept-Language": "en-US,en;q=0.5",
}


In [53]:
def fetch_page(url, timeout=30):

    response = requests.get(url, headers=HEADERS, timeout=timeout)

    if response.status_code != 200:
        print(f"Failed {url}: {response.status_code}")
        return None

    return response.text

def get_article_urls(journal_url):
    articles_url=(journal_url.rstrip("/")+"/en/articles")

    html=fetch_page(articles_url)
    if html is None: 
        return []
    soup=BeautifulSoup(html,"html.parser")

    article_urls=set()

    for a in soup.find_all("a",href=True):
        href=a["href"]
        if href.startswith("/en/articles/"):
            full_url=(journal_url.rstrip("/")+href)
            article_urls.add(full_url)

    return article_urls

def extract_dc_metadata(article_url):
    html=fetch_page(article_url)
    if html is None :
        return {}

    soup=BeautifulSoup(html,"html.parser")
    metadata=defaultdict(list)

    for meta in soup.find_all("meta"):
        name=meta.get("name")
        content=meta.get("content")

        if(name and content and name.startswith("DC.")):
            metadata[name].append(content)
    return dict(metadata)

In [ ]:
def collect_journal(journal_url):

    print(f"\nCollecting {journal_url}")

    article_urls = get_article_urls(journal_url)

    print(f"Found {len(article_urls)} articles")

    records = []

    for i, article_url in enumerate(article_urls, start=1):

        metadata = extract_dc_metadata(article_url)

        if metadata:
            records.append(
                {
                    "journal_url": journal_url,
                    "article_url": article_url,
                    "metadata": metadata,
                }
            )
    print(f"{journal_url}: {len[records]} articles ")

    return records


In [83]:
def get_journal_list(base_url="https://www.sljol.info/en/journals", max_pages=20):

    journals = []

    for page in range(1, max_pages + 1):
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}?page={page}"

        print(f"Scraping page {page}: {url}")

        html = fetch_page(url)

        if html is None:
            break

        soup = BeautifulSoup(html, "html.parser")

        page_count = 0

        for a in soup.find_all("a", href=True):
            href = a["href"]
            name = a.get_text(" ", strip=True)

            # only journal subdomains
            if "sljol.info" in href and href.count(".sljol.info") == 1:
                # remove trailing paths
                journal_url = href.split("/about")[0]

                journal_url = journal_url.rstrip("/")

                # avoid main site
                if journal_url != "https://www.sljol.info":
                    journals.append({"journal_name": name, "journal_url": journal_url})

                    page_count += 1

        

        # stop if no next page data
        if page_count == 0:
            break

    journals_df = pd.DataFrame(journals)

    journals_df = journals_df.drop_duplicates(subset=["journal_url"])

    return journals_df


In [66]:
journals_df = get_journal_list()

journals_df.head()


Scraping page 1: https://www.sljol.info/en/journals
Found 77 journals
Scraping page 2: https://www.sljol.info/en/journals?page=2
Found 75 journals
Scraping page 3: https://www.sljol.info/en/journals?page=3
Found 76 journals
Scraping page 4: https://www.sljol.info/en/journals?page=4
Found 75 journals
Scraping page 5: https://www.sljol.info/en/journals?page=5
Found 75 journals
Scraping page 6: https://www.sljol.info/en/journals?page=6
Found 77 journals
Scraping page 7: https://www.sljol.info/en/journals?page=7
Found 75 journals
Scraping page 8: https://www.sljol.info/en/journals?page=8
Found 3 journals
Scraping page 9: https://www.sljol.info/en/journals?page=9
Found 0 journals


,journal_name,journal_url
0,,https://agrieast.sljol.info
3,,https://amj.sljol.info
6,,https://anvesana.sljol.info
9,,https://aeb.sljol.info
12,,https://ajf.sljol.info


In [ ]:
print(len(journals_df))


178


In [98]:
record_collection=[]

for i in range(len(journals_df)//2):
    print(i)
    journal=journals_df.iloc[i]
    record=collect_journal(journal["journal_url"])
    record_collection.append(record)


0

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
1

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
2

Found 23 articles
1/23
2/23
3/23
4/23
5/23
6/23
7/23
8/23
9/23
10/23
11/23
12/23
13/23
14/23
15/23
16/23
17/23
18/23
19/23
20/23
21/23
22/23
23/23
3

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
4

Found 10 articles
1/10
2/10
3/10
4/10
5/10
6/10
7/10
8/10
9/10
10/10
5

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
6

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/2

KeyboardInterrupt: 

25


26

In [100]:
print(record_collection)
print(len(record_collection))

[[{'journal_url': 'https://agrieast.sljol.info', 'article_url': 'https://agrieast.sljol.info/en/articles/10.4038/agrieast.v18i1.129', 'metadata': {'DC.Creator': ['Pratheeban, S.', 'De Silva, C. S.', 'Silva, A. L. C.', 'De Silva, S.'], 'DC.Description.abstract': ['This study examines the effects of nitrogen on the growth, yield and quality of three sugarcane varieties under irrigated conditions in the low country dry zone of Sri Lanka. Three sugarcane varieties (SL96-128, SL04-624 and SL8306) were tested at four nitrogen fertilizer levels (0, 100, 200, 300 kg/ha) in a split plot design with four replicates. Selected growth, yield and quality parameters were studied in this field experiment. The results of the study demonstrated that application of 100kg/ha or 300kg/ha N fertilizer has shown highest girth of stem (cm), partitioning biomass to millable stalk (PBMS), brix and pol percentage in all three varieties, SL04-624, SL8306 and SL96-128. In addition, variety SL04-624 have shown high

In [104]:
print(len(record))
print(len(record[0]["metadata"].keys()))
keys=list(record[0]["metadata"].keys())
for i,key in enumerate(keys) :
    print(i,key)


25
26
0 DC.Creator
1 DC.Description.abstract
2 DC.date
3 DC.Date.available
4 DC.Date.created
5 DC.Date.dateAccepted
6 DC.Date.dateCopyrighted
7 DC.Date.dateSubmitted
8 DC.Description
9 DC.Format
10 DC.Relation.hasFormat
11 DC.Relation.hasVersion
12 DC.Identifier
13 DC.Identifier.identifier
14 DC.Date.issued
15 DC.Language
16 DC.Language.language
17 DC.Format.medium
18 DC.Publisher.publisher
19 DC.Source.source
20 DC.Title.title
21 DC.Type.type
22 DC.Rights.rights
23 DC.Rights.license
24 DC.rightsHolder
25 DC.Rights.rightsHolder


In [105]:
for i in range(16,50):
    print(i)
    journal = journals_df.iloc[i]
    record = collect_journal(journal["journal_url"])
    record_collection.append(record)


16

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
17

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
18

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
19

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
20

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
21

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
22

Found 25 articles


In [106]:
len(record_collection)

49

In [110]:
for i in range(50, len(journals_df)//2):
    print(i)
    journal = journals_df.iloc[i]
    record = collect_journal(journal["journal_url"])
    record_collection.append(record)

for i in range(len(journals_df)//2+1,len(journals_df)):
    print(i)
    journal = journals_df.iloc[i]
    record = collect_journal(journal["journal_url"])
    record_collection.append(record)


50

Found 23 articles
1/23
2/23
3/23
4/23
5/23
6/23
7/23
8/23
9/23
10/23
11/23
12/23
13/23
14/23
15/23
16/23
17/23
18/23
19/23
20/23
21/23
22/23
23/23
51

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
52

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
53

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
54

Found 9 articles
1/9
2/9
3/9
4/9
5/9
6/9
7/9
8/9
9/9
55

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
56

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
2

ConnectionError: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))

In [111]:
len(record_collection)

74

In [112]:
for i in range(74, len(journals_df) // 2):
    print(i)
    journal = journals_df.iloc[i]
    record = collect_journal(journal["journal_url"])
    record_collection.append(record)

for i in range(len(journals_df) // 2 + 1, len(journals_df)):
    print(i)
    journal = journals_df.iloc[i]
    record = collect_journal(journal["journal_url"])
    record_collection.append(record)


74

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
75

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
76

Found 15 articles
1/15
2/15
3/15
4/15
5/15
6/15
7/15
8/15
9/15
10/15
11/15
12/15
13/15
14/15
15/15
77

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
78

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
79

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/25
13/25
14/25
15/25
16/25
17/25
18/25
19/25
20/25
21/25
22/25
23/25
24/25
25/25
80

Found 25 articles
1/25
2/25
3/25
4/25
5/25
6/25
7/25
8/25
9/25
10/25
11/25
12/

ConnectionError: HTTPSConnectionPool(host='sljas.sljol.info', port=443): Read timed out.

In [113]:
len(record_collection)

100

In [4]:
df=pd.read_csv("../data/processed/crossref/crossref_sri_lanka_works.csv")
print(df.shape)

(16959, 34)


/tmp/ipykernel_58892/3515302436.py:1: DtypeWarning: Columns (0: volume, 1: publisher-location, 2: event.sponsor, 3: original-title) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("../data/processed/crossref/crossref_sri_lanka_works.csv")
